In [1]:
import pandas as pd
import duckdb

# 设备每日运行日志表
log_data = [
    ["A", "2026-07-01", "NORMAL", 2],
    ["A", "2026-07-02", "ERROR", 5],
    ["A", "2026-07-03", "ERROR", 6],

    ["B", "2026-07-01", "NORMAL", 1],
    ["B", "2026-07-02", "NORMAL", 2],
    ["B", "2026-07-03", "ERROR", 4],

    ["C", "2026-07-01", "ERROR", 7],
    ["C", "2026-07-02", "NORMAL", 3],

    # E 在日志表中存在，但设备信息表中没有
    ["E", "2026-07-01", "ERROR", 9],
    ["E", "2026-07-02", "ERROR", 8],
]

df_log = pd.DataFrame(
    log_data,
    columns=["device_id", "stat_date", "status", "alarm_count"]
)

df_log["stat_date"] = pd.to_datetime(df_log["stat_date"])


# 设备基础信息表
device_data = [
    ["A", "R34", "VIS", "FS11"],
    ["B", "R34", "RVR", "LT31"],
    ["C", "R35", "VIS", "FS11"],

    # D 在设备信息表中存在，但日志表中没有
    ["D", "R35", "RVR", "LT31"],
]

df_device = pd.DataFrame(
    device_data,
    columns=["device_id", "site", "device_type", "model"]
)

print("df_log:")
print(df_log)

print("\ndf_device:")
print(df_device)

df_log:
  device_id  stat_date  status  alarm_count
0         A 2026-07-01  NORMAL            2
1         A 2026-07-02   ERROR            5
2         A 2026-07-03   ERROR            6
3         B 2026-07-01  NORMAL            1
4         B 2026-07-02  NORMAL            2
5         B 2026-07-03   ERROR            4
6         C 2026-07-01   ERROR            7
7         C 2026-07-02  NORMAL            3
8         E 2026-07-01   ERROR            9
9         E 2026-07-02   ERROR            8

df_device:
  device_id site device_type model
0         A  R34         VIS  FS11
1         B  R34         RVR  LT31
2         C  R35         VIS  FS11
3         D  R35         RVR  LT31


## Task 1：找出日志表中有、设备信息表中没有的设备日志

### 找出：

- df_log 中存在，
- 但 df_device 中不存在的 device_id。

### 输出字段：

- `device_id`
- `stat_date`
- `status`
- `alarm_count`

### 业务要求：

- 只输出无法匹配到设备信息的日志记录。

- 根据当前数据，应该能找出设备 E 的日志。

In [9]:
# ==================
# Task1(SQL轨道)
# ==================

# 解法1
query1 = """
SELECT
    lo.device_id,
    lo.stat_date,
    lo.status,
    lo.alarm_count
FROM df_log AS lo
LEFT JOIN df_device AS dv
    ON lo.device_id = dv.device_id
WHERE dv.device_id IS NULL
ORDER BY lo.device_id, lo.stat_date;
"""

# 解法2
query2 = """
SELECT
    lo.device_id,
    lo.stat_date,
    lo.status,
    lo.alarm_count
FROM df_log AS lo
WHERE NOT EXISTS(
        SELECT
            1
        FROM df_device AS dv
        WHERE lo.device_id = dv.device_id
    )
ORDER BY lo.device_id, lo.stat_date;
"""

df_sql = duckdb.execute(query2).fetchdf()
df_sql

,device_id,stat_date,status,alarm_count
0,E,2026-07-01,ERROR,9
1,E,2026-07-02,ERROR,8


In [14]:
# ==================
# Task1(PANDAS轨道)
# ==================

# 解法1
df_pd1 = (
    df_log
    .merge(
        df_device,
        how='left',
        on='device_id',
        indicator=True
    )
    .loc[lambda x:x['_merge'] == 'left_only']
    [
        [
            'device_id',
            'stat_date',
            'status',
            'alarm_count'
        ]
    ]
    .sort_values(by=['device_id', 'stat_date'])
    .reset_index(drop=True)
)

# 解法2
df_pd2 = (
    df_log
    .loc[
        lambda x:~x['device_id'].isin(df_device['device_id'])
    ]
    [
        [
            'device_id',
            'stat_date',
            'status',
            'alarm_count'
        ]
    ]
    .sort_values(by=['device_id', 'stat_date'])
    .reset_index(drop=True)
)
df_pd2

,device_id,stat_date,status,alarm_count
0,E,2026-07-01,ERROR,9
1,E,2026-07-02,ERROR,8


## Task 2：找出设备信息表中有、日志表中没有的设备

### 题目要求

**找出：**

- df_device 中存在，
- 但 df_log 中从未出现过的 device_id。

### 输出字段：

- `device_id`
- `site`
- `device_type`
- `model`

### 要求：

- 只输出没有任何日志记录的设备。

In [19]:
# ==================
# Task2(SQL轨道)
# ==================

# 解法1
query1 = """

SELECT
    dv.device_id,
    dv.site,
    dv.device_type,
    dv.model
FROM df_device  AS dv
LEFT JOIN df_log AS lo
ON dv.device_id = lo.device_id
WHERE lo.device_id IS NULL
"""

# 解法2
query2 = """
SELECT
    dv.device_id,
    dv.site,
    dv.device_type,
    dv.model

FROM df_device AS dv
WHERE NOT EXISTS(
    SELECT
        1
    FROM df_log AS lo
    WHERE dv.device_id = lo.device_id
)
ORDER BY dv.device_id
"""
df_sql = duckdb.execute(query2).fetchdf()
df_sql

,device_id,site,device_type,model
0,D,R35,RVR,LT31


In [21]:
# ==================
# Task2(PANDAS轨道)
# ==================

# 解法1
df_pd1 = (
    df_device
    .merge(
        df_log,
        how='left',
        on='device_id',
        indicator=True
    )
    .loc[lambda x:x['_merge'] == 'left_only']
    [
        [
            'device_id',
            'site',
            'device_type',
            'model'
        ]
    ]
    .sort_values(by=['device_id','device_type'])
    .reset_index(drop=True)
)

# 解法2
df_pd2 = (
    df_device
    .loc[
        lambda x:~x['device_id'].isin(df_log['device_id'])
    ]
    [
        [
            'device_id',
            'site',
            'device_type',
            'model'
        ]
    ]
    .sort_values(by=['device_id','device_type'])
    .reset_index(drop=True)
)
df_pd2

,device_id,site,device_type,model
0,D,R35,RVR,LT31


## Task 3：找出从未出现过 ERROR 的设备

### 题目目标

从设备信息表 `df_device` 出发，找出：

```text
在设备信息表中登记过，
但在日志表 df_log 中从未出现过 ERROR 状态的设备。
```

---

### 输出字段

```text
device_id
site
device_type
model
```

---

### 业务要求

```text
只看 df_device 中登记过的设备。

如果某个设备在 df_log 中出现过 ERROR 状态，
则不输出。

如果某个设备只有 NORMAL 日志，
但从未出现过 ERROR，
则需要输出。

如果某个设备完全没有任何日志记录，
也算作从未出现过 ERROR，
也需要输出。
```

---

### 业务理解

本题不是找：

```text
没有任何日志的设备
```

而是找：

```text
没有 ERROR 日志的设备
```

所以设备可以分成三类：

| 情况 | 是否输出 |
|---|---|
| 出现过 ERROR 日志 | 不输出 |
| 只有 NORMAL 日志 | 输出 |
| 完全没有日志 | 输出 |

---

### 示例解释

如果设备表中有：

```text
A
B
C
D
```

日志表中：

```text
A 出现过 ERROR
B 只出现过 NORMAL
C 出现过 ERROR
D 没有任何日志
```

那么本题结果应该输出：

```text
B
D
```

原因是：

```text
B 虽然有日志，但从未 ERROR。
D 虽然没有日志，但也从未 ERROR。
```

---

### 解题要求

本题建议分别使用：

```text
SQL：NOT EXISTS
Pandas：筛选 ERROR 设备后使用 ~isin()
```

重点判断：

```text
是否存在满足 status = 'ERROR' 的日志记录。
```

而不是简单判断：

```text
是否存在任意日志记录。
```

In [24]:
# ==================
# Task3(SQL轨道)
# ==================

query = """
SELECT
    dv.device_id,
    dv.site,
    dv.device_type,
    dv.model
FROM df_device AS dv
WHERE NOT EXISTS (
    SELECT
        1
    FROM df_log AS lo
    WHERE lo.device_id = dv.device_id
      AND lo.status = 'ERROR'
)
ORDER BY dv.device_id
"""

df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,site,device_type,model
0,D,R35,RVR,LT31


In [ ]:
# ==================
# Task3(PANDAS轨道)
# ==================

error_devices = (
    df_log
    .loc[
        lambda x: x['status'] == 'ERROR',
        'device_id'
    ]
    .drop_duplicates()
)

df_pd = (
    df_device
    .loc[
        lambda x: ~x['device_id'].isin(error_devices)
    ]
    [
        [
            'device_id',
            'site',
            'device_type',
            'model'
        ]
    ]
    .sort_values(by='device_id')
    .reset_index(drop=True)
)

df_pd

,device_id,site,device_type,model
0,D,R35,RVR,LT31
